# Main S4 — faithfulness judge
**Run with: Save Version → "Save & Run All (Commit)"**, accelerator **GPU T4 x2**, Internet On. It runs unattended; if any step fails the notebook stops with `FAILED` — send me the log. When it finishes, download `main_s4_results.zip` from the version's Output tab and send it to me.

**Input:** Add Input → Your Work → the S3a notebook (its saved output with `main_s3a_results.zip`).

In [ ]:
# Cell 0 — helper: run a shell command, stream its output, STOP the notebook if it fails
import os, subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")

In [ ]:
# Cell 1 — hardware, code at tag main-v1, pinned environment, environment check, tests
sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv || true")
sh("df -h / /tmp /kaggle/working && free -g && nproc")
os.chdir("/kaggle/working")
sh("git clone --depth 1 --branch main-v1 https://github.com/adeelcheena791/rag-embed-transfer.git rag")
os.chdir("/kaggle/working/rag")
sh("git log --oneline -1")
sh("pip install -q -r requirements-retrieval-pinned.txt")
sh("pip install -q -r requirements-generation-pinned.txt")
sh("python -m src.check_env --lock requirements-generation.lock")
sh("python -m src.env_report --session main_S4 --out results/raw/env_main_S4.json")
sh("python -m pytest -q -p no:cacheprovider 2>&1 | tail -3")

In [ ]:
# Cell 2 — restore S3a outputs (they include S1's) + re-download the pinned corpus text
sh("python -m src.restore --name main_s3a_results")
sh("python -m src.data.download --config configs/main.yaml")

In [ ]:
# Cell 3 — judge (zero-shot, per P6): 7 retrievers + GOLD x first 500 queries = 4,000 judgements, TP=2 (~1 h)
sh("python -m src.evaluation.judge --config configs/main.yaml --conditions R0,R1,R2,R3,R4,R5,R6,GOLD --corpus main --n 500 --variant zero_shot --auto-eager-retry")

In [ ]:
# Last cell — summary + package (becomes this version's saved output)
sh("python -m src.pilot_summary --config configs/main.yaml")
sh("zip -qr /kaggle/working/main_s4_results.zip results data/processed configs/resolved_revisions.json logs requirements-*.lock")
sh("ls -la /kaggle/working/main_s4_results.zip")
sh("rm -rf /kaggle/working/rag/data/raw /tmp/models")   # re-downloadable; keeps the saved output small